# GraphRAG Pipeline + Evaluace

Tento notebook obsahuje **dvě části**:

1. **Pipeline** (buňky 1–12) — načtení dokumentu, build grafu, indexy, retrievery, RAG chain
2. **Evaluace** (buňky 13+) — testovací sady, metrikY (BERTScore, ROUGE-L, RAGAS), `evaluate()`

**Důležité:** Evaluace volá přímo retrievery z pipeline (`full_retriever`, `standard_retriever`, `pure_graph_retriever`), takže výsledky jsou vždy identické s produkcí.

---

##  Část 1 — Pipeline

In [ ]:
import hashlib
import inspect
import os
import pickle
import re
import threading
import time

from dotenv import load_dotenv
from IPython.display import display
from neo4j import GraphDatabase
from pydantic import BaseModel, Field

from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_experimental.llms.ollama_functions import OllamaFunctions
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


load_dotenv()

True

In [ ]:
# Připojení k Neo4j 
graph = Neo4jGraph(
    url=os.environ.get('NEO4J_URI'),
    username=os.environ.get('NEO4J_USERNAME'),
    password=os.environ.get('NEO4J_PASSWORD')
)

# Inicializace modelu 
llm = OllamaFunctions(model="llama3.1:8b", temperature=0, format="json", num_predict=4096)

In [ ]:
file_path = "../documents/md/AF_III_02_01__Strojirenska_Firma.md"
cache_file = "../cache/AF_III_02_01__Strojirenska_Firma.pkl"

In [ ]:
with open(file_path, "r", encoding="utf-8") as f:
    md_content = f.read()

# 1. Rozšířené split podle nadpisů (H1 - H6)
headers_to_split_on = [
    ("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3"),
    ("####", "Header 4"), ("#####", "Header 5"), ("######", "Header 6"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(md_content)

# 2. Zkrácení na logické celky
text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=400)
all_documents = text_splitter.split_documents(md_header_splits)

# Odstranění pomocných sekcí (obsah, literatura) — neobsahují znalostní obsah
documents = []
for doc in all_documents:
    h1 = doc.metadata.get("Header 1", "").strip().lower()
    
    # Pokud je hlavní nadpis "obsah" nebo "literatura", chunk zahodit
    if "obsah" in h1 or "zdroje" in h1 or "literatura" in h1:
        continue
        
    documents.append(doc)

print(f"Původně vytvořeno: {len(all_documents)} chunků.")
print(f"Po vyhození Obsahu zbylo k analýze: {len(documents)} čistých chunků.")

Původně vytvořeno: 794 chunků.
Po vyhození Obsahu zbylo k analýze: 662 čistých chunků.


In [ ]:
def get_chunk_hash(text):
    return hashlib.md5(text.encode('utf-8')).hexdigest()

# Výpočet MD5 hashů pro všechny chunky aktuálního dokumentu
for doc in documents:
    doc.metadata["hash"] = get_chunk_hash(doc.page_content)

# Načtení existujících hashů z Neo4j a oprava chybějících záznamů
driver = GraphDatabase.driver(
    os.environ.get('NEO4J_URI'),
    auth=(os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
)

with driver.session() as session:
    result = session.run("MATCH (d:Document) RETURN d.text AS text, d.hash AS hash")
    records = list(result)

    neo4j_hashes = set()
    updates = []

    for record in records:
        if record["hash"]:
            neo4j_hashes.add(record["hash"])
        elif record["text"]:
            # Hash chybí — dopočítá se ze zdrojového textu a zařadí do opravné dávky
            computed_hash = get_chunk_hash(record["text"])
            neo4j_hashes.add(computed_hash)
            updates.append({"text": record["text"], "hash": computed_hash})

    if updates:
        print(f"Oprava chybějících hashů: {len(updates)} dokumentů")
        session.run("""
        UNWIND $updates AS up
        MATCH (d:Document {text: up.text})
        SET d.hash = up.hash
        """, {"updates": updates})

# Odstranění zastaralých chunků z Neo4j (smazané nebo upravené části dokumentu)
current_hashes = {doc.metadata["hash"] for doc in documents}
obsolete_hashes = neo4j_hashes - current_hashes

if obsolete_hashes:
    print(f"Odstraňuji zastaralé chunky z Neo4j: {len(obsolete_hashes)}")
    with driver.session() as session:
        for obs_hash in obsolete_hashes:
            session.run("MATCH (d:Document {hash: $hash}) DETACH DELETE d", {"hash": obs_hash})
        # Odstranění osiřelých entit bez jakékoliv vazby
        session.run("""
        MATCH (e)
        WHERE (e:Hlavni_Pojem OR e:Proces_Cinnost OR e:Nastroj_System OR
               e:Metrika_Parametr OR e:Subjekt_Role OR e:Kapitola OR e:Podkapitola)
          AND NOT (e)--()
        DELETE e
        """)
    neo4j_hashes -= obsolete_hashes
    print("Zastaralé chunky a osiřelé entity odstraněny.")

driver.close()

# Načtení lokální PKL cache
local_cache = {}
if os.path.exists(cache_file):
    with open(cache_file, "rb") as f:
        local_cache = pickle.load(f)

# Synchronizace PKL cache s aktuálním stavem dokumentu
obsolete_pkl_hashes = set(local_cache.keys()) - current_hashes

if obsolete_pkl_hashes:
    print(f"Čištění PKL cache: {len(obsolete_pkl_hashes)} zastaralých záznamů")
    for obs_hash in obsolete_pkl_hashes:
        del local_cache[obs_hash]
    with open(cache_file, "wb") as f:
        pickle.dump(local_cache, f)

# Určení chunků vyžadujících LLM extrakci
missing_in_neo4j = [doc for doc in documents if doc.metadata.get("hash") not in neo4j_hashes]
docs_for_llm = [doc for doc in missing_in_neo4j if doc.metadata.get("hash") not in local_cache]

print(f"Chunky v dokumentu:   {len(documents)}")
print(f"Chunky v Neo4j:       {len(neo4j_hashes)}")
print(f"Chunky v PKL cache:   {len(local_cache)}")
print(f"Chunky ke zpracování: {len(docs_for_llm)}")

# LLM extrakce entit a vztahů — pouze pro chunky chybějící v obou úložištích
if docs_for_llm:
    print(f"Spouštím LLM extrakci: {len(docs_for_llm)} chunků")

    universal_nodes = ["Hlavni_Pojem", "Proces_Cinnost", "Nastroj_System", "Metrika_Parametr", "Subjekt_Role"]
    universal_relationships = ["SKLADA_SE_Z", "VYUZIVA", "MA_VLASTNOST", "PROVADI", "SOUVISI_S"]

    llm_transformer = LLMGraphTransformer(
        llm=llm, allowed_nodes=universal_nodes, allowed_relationships=universal_relationships
    )

    # Iterativní zpracování s průběžným ukládáním cache po každém chunku
    for i, doc in enumerate(docs_for_llm):
        doc_hash = doc.metadata["hash"]
        print(f"  {i+1}/{len(docs_for_llm)} (hash: {doc_hash[:8]})")
        try:
            extracted_graphs = llm_transformer.convert_to_graph_documents([doc])
            if extracted_graphs:
                local_cache[doc_hash] = extracted_graphs[0]
                with open(cache_file, "wb") as f:
                    pickle.dump(local_cache, f)
        except Exception as e:
            print(f"  Chyba u chunku {i+1}, přeskočeno: {e}")
            continue

    print("Extrakce dokončena.")
else:
    if missing_in_neo4j:
        print("Všechna chybějící data načtena z PKL cache, LLM extrakce není vyžadována.")
    else:
        print("Neo4j je aktuální, žádné zpracování není vyžadováno.")

# Příprava grafových dokumentů k nahrání do Neo4j
new_graph_documents = [
    local_cache[doc.metadata["hash"]]
    for doc in missing_in_neo4j
    if doc.metadata.get("hash") in local_cache
]

📊 Celkem chunků v textu: 662
🗄️ Z toho už je v Neo4j: 643
📁 Z toho už máme předpočítáno v PKL: 643
🔥 Bude se přes LLM reálně počítat: 1 chunků
⏳ Spouštím LLM extrakci pro 1 chunků...
  Zpracovávám 1/1 (Hash: 58233aa0)...
    ❌ Chyba při zpracování chunku 1 (přeskakuji): 'llama3.1:8b' did not respond with valid JSON. 
                Please try again. 
                Response: {
  "tool": "DynamicGraph",
  "tool_input": {
    "$defs": {
      "SimpleNode": [
        {
          "id": "Firemní kultura",
          "type": "Hlavni_Pojem"
        },
        {
          "id": "Metody řízení firmy",
          "type": "Proces_Cinnost"
        },
        {
          "id": "Podniková architektura",
          "type": "Nastroj_System"
        },
        {
          "id": "Podniková organizace",
          "type": "Subjekt_Role"
        },
        {
          "id": "Dislokace podniku",
          "type": "Metrika_Parametr"
        },
        {
          "id": "Úroveň podnikových procesů",
          

In [ ]:
# Nahrání extrahovaných grafových dokumentů do Neo4j
if new_graph_documents:
    print(f"Nahrávám {len(new_graph_documents)} dokumentů do Neo4j...")
    graph.add_graph_documents(
        new_graph_documents,
        baseEntityLabel=True,
        include_source=True  # Vytvoří uzel Document propojený s extrahovanými entitami
    )
    print("Nahrání dokončeno.")
else:
    print("Neo4j je aktuální, žádné nové dokumenty k nahrání.")

# Sestavení hierarchie kapitol z metadat chunků
print("Sestavuji stromovou strukturu kapitol...")

driver = GraphDatabase.driver(
    os.environ.get('NEO4J_URI'),
    auth=(os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
)

with driver.session() as session:
    for doc in documents:
        chunk_hash = doc.metadata.get("hash")

        # Sestavení cesty nadpisů H1–H6 pro daný chunk
        # Složené path_id zajišťuje jedinečnost podkapitol se stejným názvem napříč kapitolami
        path_nodes = []
        current_path_id = ""

        for i in range(1, 7):
            h = doc.metadata.get(f"Header {i}")
            if h:
                current_path_id = f"{current_path_id} | {h}" if current_path_id else h
                path_nodes.append({"id": current_path_id, "name": h, "level": i})

        if not path_nodes:
            path_nodes = [{"id": "Obecné", "name": "Obecné", "level": 1}]

        # Vytvoření uzlů Kapitola a propojení do stromové hierarchie
        for i, item in enumerate(path_nodes):
            session.run(
                """
                MERGE (k:Kapitola {id: $id})
                SET k.nazev = $name, k.uroven = $level
                """,
                {"id": item["id"], "name": item["name"], "level": item["level"]}
            )

            # Propojení podkapitoly s nadřazenou kapitolou
            if i > 0:
                parent_item = path_nodes[i - 1]
                session.run(
                    """
                    MATCH (parent:Kapitola {id: $parent_id})
                    MATCH (child:Kapitola {id: $child_id})
                    MERGE (parent)-[:OBSAHUJE_PODKAPITOLU]->(child)
                    """,
                    {"parent_id": parent_item["id"], "child_id": item["id"]}
                )

        # Propojení nejhlubšího uzlu hierarchie s textovým chunkem
        leaf_item = path_nodes[-1]
        session.run(
            """
            MATCH (d:Document {hash: $hash})
            MATCH (k:Kapitola {id: $leaf_id})
            MERGE (k)-[:OBSAHUJE_TEXT]->(d)
            """,
            {"hash": chunk_hash, "leaf_id": leaf_item["id"]}
        )

driver.close()
print("Hierarchie kapitol sestavena.")

✅ Znalosti v Neo4j jsou stoprocentně aktuální, není co přidávat.
⏳ Buduji pevnou stromovou strukturu kapitol z metadat...
✅ Hierarchie kapitol byla úspěšně vygenerována a propojena s texty a entitami!


In [ ]:
# Inicializace embeddingového modelu
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    num_ctx=8192  # Maximální kontextové okno modelu nomic-embed-text
)

# Vytvoření hybridního vektorového indexu nad uzly Document
try:
    vector_index = Neo4jVector.from_existing_graph(
        embeddings,
        search_type="hybrid",
        node_label="Document",
        text_node_properties=["text"],
        embedding_node_property="embedding"
    )
    vector_retriever = vector_index.as_retriever(search_kwargs={"k": 10})
    print("Vektorový index vytvořen.")
except Exception as e:
    print(f"Chyba při vytváření vektorového indexu: {e}")

# Vytvoření fulltextového indexu nad entitami a kapitolami
def create_fulltext_index():
    driver = GraphDatabase.driver(
        os.environ['NEO4J_URI'],
        auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD'])
    )
    with driver.session() as session:
        # Znovuvytvoření indexu — zajišťuje konzistenci při opakovaném spuštění
        session.run("DROP INDEX fulltext_entity_id IF EXISTS")
        session.run("""
        CREATE FULLTEXT INDEX fulltext_entity_id IF NOT EXISTS
        FOR (n:__Entity__|Kapitola|Podkapitola)
        ON EACH [n.id, n.nazev]
        """)
    driver.close()

create_fulltext_index()
print("Fulltextový index nad entitami a kapitolami vytvořen.")

✅ Vektorový index úspěšně vytvořen s max. kontextem.
✅ Fulltextový index pokrývající entity i kapitoly je připraven.


In [ ]:
# Inicializace embeddingového modelu pro manuální indexaci
embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text",
    num_ctx=8192  # Maximální kontextové okno modelu nomic-embed-text
)

driver = GraphDatabase.driver(
    os.environ['NEO4J_URI'],
    auth=(os.environ['NEO4J_USERNAME'], os.environ['NEO4J_PASSWORD'])
)

def build_vector_index():
    with driver.session() as session:
        # Odstranění stávajícího indexu před znovuvytvořením
        session.run("DROP INDEX vector IF EXISTS")

        # Načtení uzlů Document bez embeddingu
        result = session.run(
            "MATCH (d:Document) WHERE d.embedding IS NULL RETURN d.id as id, d.text as text"
        )
        nodes = [{"id": r["id"], "text": r["text"]} for r in result]

        print(f"Uzly ke zpracování: {len(nodes)}")

        for i, node in enumerate(nodes):
            try:
                # Text oříznut na 8 000 znaků
                safe_text = node["text"][:8000]
                vector = embeddings_model.embed_query(safe_text)
                session.run(
                    "MATCH (d:Document {id: $id}) SET d.embedding = $vector",
                    {"id": node["id"], "vector": vector}
                )
                if (i + 1) % 10 == 0:
                    print(f"  Zpracováno {i+1}/{len(nodes)}")
            except Exception as e:
                print(f"  Chyba u uzlu {node['id']}: {e}")

        # Vytvoření vektorového indexu nad existujícími embeddingy
        session.run("""
            CREATE VECTOR INDEX vector IF NOT EXISTS
            FOR (m:Document) ON (m.embedding)
            OPTIONS {indexConfig: {
              `vector.dimensions`: 768,
              `vector.similarity_function`: 'cosine'
            }}
        """)
        print("Vektorový index vytvořen.")

build_vector_index()
driver.close()

🧹 Starý index smazán.
📦 Nalezeno 0 dokumentů v DB. Spouštím individuální embedding...
✅ Vektorový index úspěšně vytvořen manuálně.


In [ ]:
# Dense retriever — kosinová podobnost (baseline Naive RAG)
neo4j_dense_index = Neo4jVector.from_existing_index(
    embeddings,
    index_name="vector",
    search_type="vector",
    node_label="Document",
    text_node_property="text"
)
dense_retriever = neo4j_dense_index.as_retriever(search_kwargs={"k": 20})

# Hybridní retriever — kosinová podobnost + BM25 (Hybrid GraphRAG)
neo4j_hybrid_index = Neo4jVector.from_existing_index(
    embeddings,
    index_name="vector",
    search_type="hybrid",
    keyword_index_name="keyword",
    node_label="Document",
    text_node_property="text"
)
hybrid_retriever = neo4j_hybrid_index.as_retriever(search_kwargs={"k": 20})

print("dense_retriever  — search_type=vector")
print("hybrid_retriever — search_type=hybrid")

✅ dense_retriever  — search_type=vector (Naive RAG)
✅ hybrid_retriever — search_type=hybrid (Hybrid GraphRAG)


In [ ]:
# Schéma pro strukturovaný výstup extrakce entit
class Entities(BaseModel):
    """Informace o klíčových entitách a pojmech v textu."""
    names: list[str] = Field(
        ...,
        description="Všechny klíčové pojmy, technické termíny, procesy a funkce. ZÁSADNÍ: Nesekej víceslovná spojení (např. 'funkce prediktivní analytiky', 'řízení dopravy') na jednotlivá slova!",
    )

# Inicializace modelů
# llm_extraction bez num_predict — parametr je relevantní pouze pro LLMGraphTransformer
# llm_rag s rozšířeným kontextem num_ctx=32768 pro zpracování dlouhého retrieval kontextu
llm_extraction = OllamaFunctions(model="llama3.1:8b", temperature=0, format="json")
llm_rag = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    num_ctx=32768
)

# Few-shot prompt pro extrakci entit z uživatelského dotazu
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """Jsi expert na extrakci klíčových pojmů z odborných dotazů pro vyhledávání v databázi.
Tvým úkolem je z uživatelské otázky vyextrahovat hlavní entity.

ZÁSADNÍ PRAVIDLO: Odborné pojmy, víceslovné názvy, vlastnosti, metriky nebo procesy ponechávej v celku! Nesekej ustálená spojení na jednotlivá slova.

PŘÍKLADY SPRÁVNÉHO CHOVÁNÍ:
Otázka: "Co je to sekvenční diagram?"
Entity: ["sekvenční diagram"]

Otázka: "Jaké jsou fáze buněčného dělení v biologii?"
Entity: ["fáze buněčného dělení", "biologie"]

Otázka: "Co patří pod funkce prediktivní analytiky v marketingu?"
Entity: ["funkce prediktivní analytiky v marketingu"]

Otázka: "Jaké jsou výhody agilního řízení projektů?"
Entity: ["výhody agilního řízení projektů"]
"""),
    ("human", "{question}")
])

# Extrakční pipeline s validací výstupu přes Pydantic schéma
entity_chain = extraction_prompt | llm_extraction.with_structured_output(Entities)


In [ ]:
def graph_enhanced_retriever(question: str):
    try:
        entities = entity_chain.invoke({"question": question})
        print(f"Extrahované entity: {entities.names}")

        all_documents = []
        seen_docs = set()
        search_queries = []

        relations_text = ""
        seen_relations = set()

        # Sestavení Lucene dotazů z extrahovaných entit
        for entity in entities.names:
            words = [w.lower() for w in entity.split() if len(w) >= 3]
            if words:
                lucene_query = " AND ".join([f"*{w[:5]}*" for w in words])
                search_queries.append(lucene_query)

        # Doplňkový dotaz sestavený přímo ze slov otázky
        question_words = [w.lower() for w in re.findall(r'\w+', question) if len(w) >= 3]
        if question_words:
            direct_query = " OR ".join([f"*{w[:5]}*" for w in question_words])
            search_queries.append(direct_query)

        search_queries = list(set(search_queries))

        for lucene_query in search_queries:
            printed_nodes = set()

            
            response = graph.query(
                """
                CALL db.index.fulltext.queryNodes('fulltext_entity_id', $lucene_query, {limit: 5})
                YIELD node

                // Traversal dolů pouze pro listové uzly bez přímých podkapitol
                OPTIONAL MATCH (node)-[:OBSAHUJE_PODKAPITOLU]->(direct_child)
                WITH node, count(direct_child) AS child_count

                OPTIONAL MATCH (node)-[:OBSAHUJE_PODKAPITOLU*1..5]->(sub_k)
                    WHERE child_count = 0

                // Traversal nahoru zachytí preamble chunky na úrovni H1
                OPTIONAL MATCH (parent_k:Kapitola)-[:OBSAHUJE_PODKAPITOLU*1..3]->(node)

                WITH node,
                     collect(DISTINCT sub_k) + [node] + collect(DISTINCT parent_k) AS all_nodes
                UNWIND all_nodes AS search_node

                MATCH (doc:Document)
                WHERE (doc)-[:MENTIONS]->(search_node) OR (search_node)-[:OBSAHUJE_TEXT]->(doc)

                OPTIONAL MATCH (doc)-[:MENTIONS]->(kapitola_entity:__Entity__)
                WHERE labels(node)[0] = 'Kapitola'

                OPTIONAL MATCH (node)-[r]-(neighbor)
                WHERE type(r) <> 'MENTIONS' AND type(r) <> 'OBSAHUJE_TEXT' AND type(r) <> 'OBSAHUJE_PODKAPITOLU'

                RETURN DISTINCT
                    node.id AS entity_name,
                    labels(node)[0] AS node_type,
                    type(r) AS relation,
                    neighbor.id AS related_entity,
                    kapitola_entity.id AS podrizena_entita,
                    doc.text AS source_doc
                LIMIT 50
                """,
                {"lucene_query": lucene_query}
            )

            for r in response:
                node_name = r.get('entity_name')
                node_type = r.get('node_type', 'neznámý typ')
                rel = r.get('relation')
                neighbor = r.get('related_entity')
                podrizena_entita = r.get('podrizena_entita')

                if node_name not in printed_nodes:
                    print(f"  Nalezen uzel [{node_type}]: '{node_name}'")
                    printed_nodes.add(node_name)

                if rel and neighbor:
                    rel_str = f"Položka '{neighbor}' (vztah ke '{node_name}': {rel})"
                    if rel_str not in seen_relations:
                        relations_text += f"- {rel_str}\n"
                        seen_relations.add(rel_str)

                if podrizena_entita:
                    sub_ent_str = f"Kapitola '{node_name}' obsahuje entitu: '{podrizena_entita}'"
                    if sub_ent_str not in seen_relations:
                        relations_text += f"- {sub_ent_str}\n"
                        seen_relations.add(sub_ent_str)

                doc = r.get('source_doc')
                if doc and doc not in seen_docs and len(doc) > 100:
                    all_documents.append(doc)
                    seen_docs.add(doc)

        print(f"Nalezeno dokumentů z grafu: {len(all_documents)}")
        return relations_text, all_documents

    except Exception as e:
        print(f"Chyba v grafovém retrieveru: {e}")
        return "", []

In [ ]:
# Cross-encoder reranker pro přeřazení dokumentů podle relevance k dotazu
# Použit multilingvální model trénovaný na datasetu mMARCO (včetně češtiny)
try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
    RERANKER_AVAILABLE = True
    print("Cross-encoder načten: mmarco-mMiniLMv2-L12-H384-v1")
except ImportError:
    RERANKER_AVAILABLE = False
    print("sentence-transformers neni dostupne, reranking preskocen")


def rerank_docs(question: str, docs, top_k: int = 10, label: str = ""):
    if not docs or not RERANKER_AVAILABLE:
        return docs[:top_k]

    # Podpora pro Document objekty i prosté řetězce
    texts = [doc.page_content if hasattr(doc, "page_content") else doc for doc in docs]
    pairs = [(question, text) for text in texts]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)

    print(f"  Reranking {label}({len(docs)} dokumentů → top {top_k}):")
    print(f"  {'Pořadí':<7} {'Skóre':>7}  Náhled")
    print("  " + "-" * 70)
    for rank_i, (score, doc) in enumerate(ranked, 1):
        text = doc.page_content if hasattr(doc, "page_content") else doc
        preview = text[:120].replace("\n", " ")
        marker = "+" if rank_i <= top_k else "-"
        print(f"  {marker} #{rank_i:<3} {score:>7.3f}  {preview}...")
    print()

    return [doc for _, doc in ranked[:top_k]]

✅ Multilingual cross-encoder načten: mmarco-mMiniLMv2-L12-H384-v1
   Podporuje češtinu (trénováno na mMARCO multilingválním datasetu).


In [ ]:
def full_retriever(question: str):
    # Vektorové vyhledávání — hybridní retriever (kosinová podobnost + BM25, k=20)
    vector_docs = hybrid_retriever.invoke(question)
    print(f"Vektorové vyhledávání: {len(vector_docs)} dokumentů")
    print("-" * 60)
    for i, doc in enumerate(vector_docs):
        meta = {k: v for k, v in doc.metadata.items() if k != "hash" and v}
        meta_str = " | ".join(str(v) for v in meta.values()) if meta else "(bez metadat)"
        preview = doc.page_content[:200].replace("\n", " ")
        print(f"  [{i+1}] {meta_str}")
        print(f"        {preview}...")
    print("-" * 60)

    graph_relations_text, graph_docs_list = graph_enhanced_retriever(question)
    print(f"Grafové vyhledávání: {len(graph_docs_list)} dokumentů")
    print("-" * 60)
    for i, doc_text in enumerate(graph_docs_list):
        preview = doc_text[:200].replace("\n", " ")
        print(f"  [{i+1}] {preview}...")
    print("-" * 60)

    reranked_vector = rerank_docs(question, vector_docs, top_k=10, label="[vector] ")
    reranked_graph = rerank_docs(question, graph_docs_list, top_k=10, label="[graph] ") if graph_docs_list else []

    # Výpis finálního pořadí dokumentů předaných LLM
    print("Finální pořadí — vektorové dokumenty:")
    print("  " + "-" * 70)
    for rank_i, doc in enumerate(reranked_vector, 1):
        text = doc.page_content if hasattr(doc, "page_content") else doc
        meta = {k: v for k, v in doc.metadata.items() if k != "hash" and v} if hasattr(doc, "metadata") else {}
        meta_str = " | ".join(str(v) for v in meta.values()) if meta else "(bez metadat)"
        preview = text[:150].replace("\n", " ")
        print(f"  #{rank_i:<3} {meta_str}")
        print(f"        {preview}...")
    print()

    if reranked_graph:
        print("Finální pořadí — grafové dokumenty:")
        print("  " + "-" * 70)
        for rank_i, doc in enumerate(reranked_graph, 1):
            text = doc.page_content if hasattr(doc, "page_content") else doc
            preview = text[:150].replace("\n", " ")
            print(f"  #{rank_i:<3} {preview}...")
        print()

    vector_text = "\n\n".join(
        doc.page_content if hasattr(doc, "page_content") else doc
        for doc in reranked_vector
    )
    graph_text = "\n\n".join(reranked_graph)

    # Ořez kontextu — llama3.1:8b spolehlivě zpracuje přibližně 10 000 znaků
    graph_text_trimmed  = graph_text[:8000]
    vector_text_trimmed = vector_text[:3000]
    graph_rel_trimmed   = graph_relations_text[:1500]

    # Grafové dokumenty jsou primárním zdrojem — řazeny na první pozici v kontextu
    context = (
        f"### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚTE CELÉ, ODPOVĚĎ JE ZDE):\n{graph_text_trimmed}"
        f"\n\n### 2. VEKTOROVÉ DOKUMENTY (Doplňující textový kontext):\n{vector_text_trimmed}"
        f"\n\n### 3. VZTAHY Z GRAFU (Strukturální kontext):\n{graph_rel_trimmed}"
    )

    # Náhled kontextu odeslaného do LLM
    char_count = len(context)
    sep = "=" * 70
    print(f"Kontext odeslaný do LLM ({char_count:,} znaků):")
    print(sep)
    print("[Sekce 1 — grafové dokumenty]")
    print(graph_text_trimmed[:800].replace("\n\n", "\n") + ("…" if len(graph_text_trimmed) > 800 else ""))
    print()
    print("[Sekce 2 — vektorové dokumenty]")
    print(vector_text_trimmed[:800].replace("\n\n", "\n") + ("…" if len(vector_text_trimmed) > 800 else ""))
    print()
    print("[Sekce 3 — vztahy z grafu]")
    print(graph_rel_trimmed[:800] + ("…" if len(graph_rel_trimmed) > 800 else ""))
    print(sep)

    return context

In [ ]:
template = """

Jsi expertní analytický systém pracující s technickou dokumentací.

### KRITICKÁ PRAVIDLA:
1. ODPOVÍDEJ POUZE NA ZÁKLADĚ DODANÉHO KONTEXTU.
2. Pro výčty použij odrážkový seznam, pro definice odstavce.
3. KOMPLETNÍ VÝČET BEZ ZKRACOVÁNÍ — žádné "a další", "apod.", "atd.".

### KONTEXT:
{context}

### OTÁZKA:
{question}


### ODPOVĚĎ:"""

prompt = ChatPromptTemplate.from_template(template)

# Sestavení RAG chain: retrieval → prompt → LLM → parsování výstupu
chain = (
    {"context": full_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm_rag
    | StrOutputParser()
)

print("RAG chain připraven.")

✅ RAG chain s přísným promptem vytvořen!


### Rychlý test pipeline
Otázku pro ověření, že vše funguje.

In [59]:
print(chain.invoke("Jake jsou klíčové aktivity v evidenci TPV?"))


--- HLEDÁM ODPOVĚĎ NA: 'Jake jsou klíčové aktivity v evidenci TPV?' ---

📚 VEKTOROVÉ VYHLEDÁVÁNÍ — nalezeno 20 dokumentů (před rerankingem):
------------------------------------------------------------
  [1] 📂 11.1 Přehled a obsah úloh řízení majetku | 11. Řízení majetku | 11.1.1 Evidence majetku
       💬 Účelem je aktualizace základních údajů majetkových databází, klasifikace majetku, technických údajů apod. (viz další obrázek).   *Obrázek 11-2: Evidence majetku*   **Zajišťuje tyto klíčové aktivity :*...
  [2] 📂 17.1 Přehled a obsah úloh OŘV | 17. Operativní řízení výroby, OŘV | 17.1.1 OŘV: Operativní evidence výroby
       💬 Účelem je vytvářet a zajistit průběžnou operativní evidenci o výrobě, a to (viz další obrázek):   - evidenci plánované a realizované výroby v kusech, - spotřebu normohodin, - spotřebu materiálu, - sor...
  [3] 📂 17.2 OŘV v kontextu řízení firmy | 17. Operativní řízení výroby, OŘV | 17.2.1 Vstupy do OŘV
       💬 Podstatné vstupy do operativního řízení výroby z os

In [ ]:
def showGraph():
    driver = GraphDatabase.driver(
        uri=os.environ.get('NEO4J_URI'),
        auth=(os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
    )

    with driver.session() as session:
        query = """
        MATCH (s)-[r]->(t)
        RETURN s, r, t
        """
        graph_data = session.run(query).graph()

    driver.close()

    widget = GraphWidget(graph=graph_data)

    def custom_node_label(node):
        props = node.get("properties", {})
        labels = node.get("labels", [])

        # Uzly Document zobrazí začátek textu místo hash identifikátoru
        if "Document" in labels:
            text = props.get("text", "")
            return text[:40] + "..." if text else "Textový blok"

        # Kapitoly a entity zobrazí své id
        return props.get("id", "Neznámý uzel")

    widget.node_label_mapping = custom_node_label
    return widget

showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

---

##  Část 2 — Evaluace

Evaluace **přímo volá retrievery z pipeline** výše — žádná duplicita kódu, žádný rozdíl v odpovědích.

**Retrievery:**
- `Standard RAG` — pouze vektorové vyhledávání (`vector_retriever`, k=20)
- `Pure Graph RAG` — pouze grafové vyhledávání + reranking
- `Hybrid GraphRAG` — kombinace grafu + vektoru + reranking (= `full_retriever`)

**Metriky:**
- `BERTScore` P/R/F1 — sémantická podobnost s ground truth
- `ROUGE-L` P/R/F1 — lexikální překryv s ground truth
- `RAGAS` Faithfulness, AnswerRelevancy, ContextRecall, AnswerCorrectness (LLM judge: Qwen)

In [ ]:
try:
    from bert_score import score as bert_score
    BERTSCORE_AVAILABLE = True
    print("BERTScore načten.")
except ImportError:
    BERTSCORE_AVAILABLE = False
    print("bert-score není dostupný: pip install bert-score")

try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
    print("ROUGE-L načten.")
except ImportError:
    ROUGE_AVAILABLE = False
    print("rouge-score není dostupný: pip install rouge-score")

try:
    from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextRecall
    from ragas import evaluate as evaluate_ragas
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from datasets import Dataset
    RAGAS_AVAILABLE = True
    print("RAGAS načten.")
except ImportError:
    RAGAS_AVAILABLE = False
    print("ragas není dostupný: pip install ragas datasets")

# LLM judge pro RAGAS — nezávislý model oddělený od modelu generujícího odpovědi
llm_judge = ChatOllama(model="qwen2.5:7b", temperature=0, num_ctx=16384)

print("Evaluační závislosti načteny.")
print("LLM judge: qwen2.5:7b")

✅ BERTScore loaded
✅ ROUGE-L loaded
✅ RAGAS loaded
✅ Eval imports hotové
   LLM judge (RAGAS): qwen2.5:7b


/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_55056/1775939859.py:22: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextRecall
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_55056/1775939859.py:22: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextRecall
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_55056/1775939859.py:22: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.m

In [ ]:
def naive_rag(question: str) -> str:
    """Naive RAG — Dense Retriever (cosine similarity).Baseline.
    """
    docs = dense_retriever.invoke(question)
    return '\n\n'.join(d.page_content for d in docs)


def graph_rag(question: str) -> str:
    """Graph RAG — Knowledge Graph Retriever"""
    rel, docs = graph_enhanced_retriever(question)
    ranked = rerank_docs(question, docs, top_k=10) if docs else []
    graph_text = '\n\n'.join(ranked)
    return (
        f'### DOKUMENTY Z GRAFU:\n{graph_text[:8000]}'
        f'\n\n### VZTAHY Z GRAFU:\n{rel[:3000]}'
    )


def hybrid_graphrag(question: str) -> str:
    """Hybrid GraphRAG — Knowledge Graph + (cosine similarity).
    Kombinuje grafovou navigaci s Hybrid Retrieverem = plný Hybrid RAG přístup.
    """
    return full_retriever(question)


SCENARE = {
    'Naive RAG':       naive_rag,       # Dense Retriever,
    'Graph RAG':       graph_rag,       # KG Retriever,
    'Hybrid GraphRAG': hybrid_graphrag, # KG + Hybrid Retriever
}


print("Evaluační scénáře připraveny:")
print("  Naive RAG       — dense retriever, bez rerankingu (baseline)")
print("  Graph RAG       — KG traversal s rerankingem")
print("  Hybrid GraphRAG — KG traversal + hybridní retriever s rerankingem")

✅ Evaluační retrievery připraveny
   Scénáře: ['Naive RAG', 'Graph RAG', 'Hybrid GraphRAG']

Naive RAG       — Dense only, BEZ rerankingu  → baseline (Lewis 2020)
Graph RAG       — KG Retriever + reranking    → přínos grafu
Hybrid GraphRAG — KG + Hybrid + reranking     → hlavní systém


###  Kompetenční otázky

Každá sada je seznam dvojic `(otázka, ground_truth)`.

In [ ]:
sada_strojirenska_firma = [
 
    # ── VÝČTOVÉ (3) ────────────────────────────────────────────────────────────
    # Testují úplnost retrieval — graf by měl vrátit celý výčet z kapitoly
 
    (
        'Jake jsou klíčové aktivity v evidenci TPV?',
        '- Evidence kusovníků, kusovníkových položek a kusovníkových vazeb:\n'
        '    - Stavebnicový kusovník\n'
        '    - Strukturní analytický kusovník\n'
        '    - Strukturní přehled\n'
        '    - Inverzní kusovník na celý sortiment\n'
        '    - Souhrnný kusovník\n'
        '- Specifikace technologických postupů\n'
        '- Vytváření a aktualizace normativní základny\n'
        '- Specifikace výrobních středisek\n'
        '- Evidence výkresů a rozpisek materiálů\n'
        '- Průběžné sledování pracnosti probíhajících výrobních zakázek'
    ),
 
    (
        'Jaké funkce zahrnuje pokročilá analytika v marketingu?',
        '- Segmentace zákazníků podle různých charakteristik '
        '(demografické, behaviorální, geografické)\n'
        '- Predikce odchodu zákazníků (churn management)\n'
        '- Credit scoring — ohodnocení zákazníka podle úvěrového rizika\n'
        '- Analýza nákupního košíku — zjišťování souvislostí mezi produkty '
        'které zákazníci kupují společně'
    ),
 
    (
        'Jaké jsou klíčové aktivity analytické úlohy v controllingu?',
        '- Analýzy finančních ukazatelů podle vybraných dimenzí\n'
        '- Analýzy obchodních ukazatelů podle vybraných dimenzí\n'
        '- Analýzy personálních ukazatelů podle vybraných dimenzí\n'
        '- Analýzy majetkových a investičních ukazatelů podle vybraných dimenzí\n'
        '- Časové analýzy v controllingu\n'
        '- Analýzy vnitropodnikových normativů\n'
        '- Vyhodnocování kalkulací'
    ),
 
    # ── DEFINICE / ÚČEL (3) ────────────────────────────────────────────────────
    # Testují přesnost — jednodušší retrieval, dobrý pro baseline
 
    (
        'Jaký je účel TPV v technické přípravě?',
        'Účelem TPV je připravit kvalitní technickou a technologickou dokumentaci výrobků, '
        'připravit podklady pro nákup na základě objemu výroby a příslušných kusovníků, '
        'připravit podklady pro řízení kooperací s výrobními partnery a udržovat celý systém norem, '
        'technické dokumentace a dokumentace technologických postupů pro řízení výroby.'
    ),
 
    (
        'Jaký je účel dílenského řízení výroby (DŘV)?',
        '- Zajišťovat realizaci výrobních zakázek podle operativních plánů výroby '
        'a dílenských plánů výroby\n'
        '- Dosahovat efektivního využití výrobních kapacit\n'
        '- Zajišťovat požadovanou kvalitu výroby a finálních výrobků'
    ),
 
    (
        'Jaký je účel controllingu?',
        'Účelem controllingu je koordinace systému řízení pro zajištění vnitřní a vnější '
        'harmonizace a zajištění informací. Controlling doplňuje a integruje management '
        'jak v koncepčním, funkcionálním a institucionálním smyslu, tak i v personálním smyslu.'
    ),
 
    # ── VZTAHOVÉ — vstupy/výstupy mezi oblastmi (3) ───────────────────────────
    # Testují traversal přes hrany grafu — silná stránka GraphRAG
    # Naive RAG tyto otázky pravděpodobně špatně zodpoví (informace jsou ve více chunkcích)
 
    (
        'Jaké jsou vstupy do TPV v oblasti plánování výrobních zakázek?',
        '- Plán výrobních zakázek\n'
        '- Plánovací sešit výroby\n'
        '- Monitorování a koordinace zakázek\n'
        '- Dostupnost zásob v čase\n'
        '- Evidence výrobních zakázek\n'
        '- Kalkulace zakázek\n'
        '- Průběžné sledování pracnosti\n'
        '- Dokumenty probíhajících, plánovaných a realizovaných zakázkách\n'
        '- Výrobní plány - dlouhodobé, střednědobé, operativní'
    ),
 
    (
        'Jaké jsou výstupy z TPV do OŘV?',
        '- Kusovníky\n'
        '- Kusovníkové položky\n'
        '- Technologické postupy\n'
        '- Normy, normativní základna\n'
        '- Výrobní střediska\n'
        '- Výkresy\n'
        '- Rozpisky materiálů\n'
        '- Technická a výrobní dokumentace'
    ),
 
    (
        'Jaké jsou vstupy do DŘV z TPV?',
        '- Normy, normativní základna\n'
        '- Kusovníky\n'
        '- Kusovníkové položky\n'
        '- Technologické postupy\n'
        '- Výrobní střediska\n'
        '- Výkresy\n'
        '- Rozpisky materiálů\n'
        '- Technická a výrobní dokumentace'
    ),
 
    # ── FAKTICKÉ — jednoduché definice (3) ────────────────────────────────────
    # Testují základní retrieval — baseline by měl zvládnout
    # Pokud baseline selže i zde, je to silný argument pro GraphRAG
 
    (
        'Co je to strukturní analytický kusovník?',
        'Strukturní analytický kusovník je celá struktura výrobku nebo vyráběné součásti '
        'a zachycuje všechny přímo i nepřímo vstupující položky. Vstupující součásti jsou '
        'zobrazeny tolikrát, kolikrát se vyskytují ve struktuře, je zde také určeno množství '
        'nižší součásti do vyšší podle výrobních.'
    ),
 
    (
        'Jaké jsou základní charakteristiky OŘV?',
        '- Využívá celé škály normativních i skutečných dat jako jsou technickohospodářské '
        'normy, kapacity pracovišť a další\n'
        '- Základním obdobím je kvartál, klouzavě se posunující v čase\n'
        '- Zakládá se na hodnocení průběžné doby výroby, hospodářských smluv a dalších parametrech\n'
        '- Zahrnuje operativní evidence, operativní hodnocení průběhu výroby i operativní '
        'plánování výroby\n'
        '- Podstatnou součástí je řízení změn a nastavení změnového řízení'
    ),
 
    (
        'Jaké KPI se využívají v technické přípravě výroby?',
        '- Technickohospodářské normy spotřeby materiálu\n'
        '- Normy zásob\n'
        '- Normy ztrát a mank\n'
        '- Kapacitní normy\n'
        '- Normy spotřeby času'
    ),
 
]
print(f'   sada_strojirenska_firma:   {len(sada_strojirenska_firma)} otázek')

   sada_strojirenska_firma:   12 otázek


In [ ]:
import threading
import inspect

LLM_TIMEOUT = 180


def compute_bertscore(predictions, references):
    """Vrátí (Precision, Recall, F1) listy pro BERTScore."""
    if not BERTSCORE_AVAILABLE:
        none = [None] * len(predictions)
        return none, none, none
    P, R, F1 = bert_score(
        predictions, references,
        model_type='bert-base-multilingual-cased', lang='cs',
        verbose=False
    )
    return P.tolist(), R.tolist(), F1.tolist()


def compute_rouge(predictions, references):
    """Vrátí (Precision, Recall, F1) listy pro ROUGE-L."""
    if not ROUGE_AVAILABLE:
        none = [None] * len(predictions)
        return none, none, none
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    P, R, F1 = [], [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)['rougeL']
        P.append(s.precision); R.append(s.recall); F1.append(s.fmeasure)
    return P, R, F1


def call_with_timeout(retriever, llm, question, timeout=LLM_TIMEOUT):
    """Zavolá retriever a LLM s timeoutem. Vrátí (answer, context, elapsed_s)."""
    result = {}

    # Prompt identický s pipeline — zajišťuje konzistenci mezi evaluací a produkcí
    EVAL_PROMPT = """
Jsi expertní analytický systém pracující s technickou dokumentací.

### KRITICKÁ PRAVIDLA:
1. ODPOVÍDEJ POUZE NA ZÁKLADĚ DODANÉHO KONTEXTU.
2. Pro výčty použij odrážkový seznam, pro definice odstavce.
3. KOMPLETNÍ VÝČET BEZ ZKRACOVÁNÍ — žádné "a další", "apod.", "atd.".

### KONTEXT:
{context}

### OTÁZKA:
{question}

### ODPOVĚĎ:"""

    def worker():
        try:
            context = retriever(question)
            prompt  = EVAL_PROMPT.format(context=context, question=question)
            answer  = llm_rag.invoke(prompt).content
            result['ok'] = (answer, context)
        except Exception as e:
            result['error'] = str(e)

    t  = threading.Thread(target=worker)
    t0 = time.time()
    t.start()
    t.join(timeout=timeout)
    elapsed = round(time.time() - t0, 1)

    if t.is_alive():
        return None, None, elapsed
    if 'error' in result:
        return f"ERROR: {result['error']}", '', elapsed
    answer, context = result['ok']
    return answer, context, elapsed


def evaluate(sada, document_name=None, timeout=LLM_TIMEOUT, mode='combined'):
    """
    Spustí evaluaci pro danou testovací sadu.

    Retrievery jsou volány přímo z pipeline — bez duplicity kódu,
    chování je identické se Streamlit aplikací.

    Parametry
    ----------
    sada          : list of (question, ground_truth)
    document_name : str, volitelný — název pro výstupní CSV
    timeout       : int, sekundy na jedno LLM volání (výchozí 180)
    mode          : 'combined' | 'reference' | 'ragas'
        'combined'   — BERTScore + ROUGE-L + RAGAS (výchozí)
        'reference'  — pouze BERTScore + ROUGE-L, bez LLM judge
        'ragas'      — pouze RAGAS
    """
    if not sada:
        print('Testovací sada je prázdná.')
        return None

    if document_name is None:
        frame = inspect.currentframe().f_back
        document_name = next(
            (k for k, v in frame.f_locals.items() if v is sada), 'document'
        ).replace('sada_', '')

    print(f'Evaluace: {document_name}')
    print(f'Otázky: {len(sada)} | Metody: {len(SCENARE)} | '
          f'LLM volání celkem: {len(sada) * len(SCENARE)}')
    print(f'Timeout: {timeout}s | Mode: {mode}\n')

    rows     = []
    out_file = f'../evaluation/evaluation_{document_name}.csv'

    for i, (question, ground_truth) in enumerate(sada, 1):
        print(f'[{i}/{len(sada)}] {question}')
        for method_name, retriever in SCENARE.items():
            print(f'  {method_name}...', end=' ', flush=True)

            answer, context, elapsed = call_with_timeout(retriever, llm_rag, question, timeout)

            if answer is None:
                print(f'Timeout ({elapsed}s), přeskočeno.')
                answer = 'TIMEOUT'; context = ''
            else:
                print(f'{elapsed}s')
                print(f'    {answer[:120].replace(chr(10), " ")}...')

            if answer != 'TIMEOUT' and mode in ('combined', 'reference'):
                bp, br, bf1 = compute_bertscore([answer], [ground_truth])
                rp, rr, rf1 = compute_rouge([answer], [ground_truth])
                bert_p, bert_r, bert_f1 = bp[0], br[0], bf1[0]
                rouge_p, rouge_r, rouge_f1 = rp[0], rr[0], rf1[0]
                print(f'    BERT F1: {round(bert_f1, 3)} | '
                      f'ROUGE-L R: {round(rouge_r, 3)} | '
                      f'ROUGE-L F1: {round(rouge_f1, 3)}')
            else:
                bert_p = bert_r = bert_f1 = None
                rouge_p = rouge_r = rouge_f1 = None

            rows.append({
                'document':     document_name,
                'method':       method_name,
                'question':     question,
                'answer':       answer,
                'context':      context,
                'time_s':       elapsed,
                'ground_truth': ground_truth,
                'BERTScore_P':  bert_p,
                'BERTScore_R':  bert_r,
                'BERTScore_F1': bert_f1,
                'ROUGE_L_P':    rouge_p,
                'ROUGE_L_R':    rouge_r,
                'ROUGE_L_F1':   rouge_f1,
            })

        # Průběžné ukládání po každé otázce
        existing = pd.read_csv(out_file) if os.path.exists(out_file) else pd.DataFrame()
        pd.concat([existing, pd.DataFrame(rows)]).drop_duplicates(
            subset=['method', 'question'], keep='last'
        ).to_csv(out_file, index=False, encoding='utf-8-sig')
        print(f'  Uloženo: {out_file}\n')

    df = pd.DataFrame(rows)

    print('\n' + '=' * 60)
    print('Vygenerované odpovědi')
    print('=' * 60)
    for _, r in df.iterrows():
        print(f'\n[{r["method"]}] {r["question"]}')
        print('-' * 40)
        print(r['answer'])

    df_valid = df[df['answer'] != 'TIMEOUT'].copy()
    if df_valid.empty:
        print('Žádné platné výsledky — všechna volání překročila timeout.')
        return df

    if mode in ('combined', 'ragas') and RAGAS_AVAILABLE:
        print('\nRAGAS evaluace...')
        ragas_llm = LangchainLLMWrapper(llm_judge)
        ragas_emb = LangchainEmbeddingsWrapper(embeddings)

        dataset = Dataset.from_dict({
            'user_input':         df_valid['question'].tolist(),
            'response':           df_valid['answer'].tolist(),
            'retrieved_contexts': [[k] for k in df_valid['context'].tolist()],
            'reference':          df_valid['ground_truth'].tolist(),
        })
        ragas_result = evaluate_ragas(
            dataset=dataset,
            metrics=[Faithfulness(), AnswerRelevancy(), ContextRecall(), AnswerCorrectness()],
            llm=ragas_llm, embeddings=ragas_emb,
            run_config=RunConfig(timeout=300, max_workers=1)
        )
        df_ragas   = ragas_result.to_pandas()
        wanted     = ['faithfulness', 'answer_relevancy', 'context_recall', 'answer_correctness']
        available  = [c for c in wanted if c in df_ragas.columns]
        ragas_cols = df_ragas[available].reset_index(drop=True)
    else:
        ragas_cols = pd.DataFrame({
            'faithfulness':       [None] * len(df_valid),
            'answer_relevancy':   [None] * len(df_valid),
            'context_recall':     [None] * len(df_valid),
            'answer_correctness': [None] * len(df_valid),
        })

    ref_cols = ['BERTScore_P', 'BERTScore_R', 'BERTScore_F1',
                'ROUGE_L_P', 'ROUGE_L_R', 'ROUGE_L_F1']
    df_out = pd.concat([
        df_valid[['document', 'method', 'question', 'answer', 'context',
                  'time_s', 'ground_truth'] + ref_cols].reset_index(drop=True),
        ragas_cols
    ], axis=1)

    metrics_cols = [c for c in ref_cols + ['faithfulness', 'answer_relevancy',
                    'context_recall', 'answer_correctness'] if c in df_out.columns]

    print(f'\nVýsledky per otázka — {document_name}')
    display(df_out[['method', 'question'] + metrics_cols].round(3))

    print(f'\nPrůměrné skóre — {document_name}')
    display(df_out.groupby('method')[metrics_cols].mean().round(3))

    if 'ROUGE_L_R' in df_out.columns and df_out['ROUGE_L_R'].notna().any():
        best_rouge = df_out.groupby('method')['ROUGE_L_R'].mean().idxmax()
        print(f'\nNejlepší ROUGE-L Recall (úplnost):    {best_rouge}')
    if 'BERTScore_F1' in df_out.columns and df_out['BERTScore_F1'].notna().any():
        best_bert = df_out.groupby('method')['BERTScore_F1'].mean().idxmax()
        print(f'Nejlepší BERTScore F1 (sémantika):     {best_bert}')
    if 'answer_correctness' in df_out.columns and df_out['answer_correctness'].notna().any():
        best_ac = df_out.groupby('method')['answer_correctness'].mean().idxmax()
        print(f'Nejlepší Answer Correctness (RAGAS):   {best_ac}')

    existing = pd.read_csv(out_file) if os.path.exists(out_file) else pd.DataFrame()
    pd.concat([existing, df_out]).drop_duplicates(
        subset=['method', 'question'], keep='last'
    ).to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f'\nVýsledky uloženy: {out_file}')
    return df_out


print('evaluate() připravena.')
print('  evaluate(sada_strojirenska_firma)                     # combined')
print('  evaluate(sada_strojirenska_firma, mode="reference")   # bez RAGAS')
print('  evaluate(sada_strojirenska_firma, mode="ragas")       # pouze RAGAS')
print('  evaluate(sada_strojirenska_firma[:1])                 # jedna otázka')

✅ evaluate() připravena

Použití:
  evaluate(sada_strojirenska_firma)                            # combined
  evaluate(sada_strojirenska_firma, mode="reference")          # bez RAGAS
  evaluate(sada_strojirenska_firma, mode="ragas")              # pouze RAGAS
  evaluate(sada_strojirenska_firma[:1], document_name="q1")    # 1 otázka


###  Spuštění evaluace

In [75]:
evaluate(sada_strojirenska_firma, timeout=120, mode='combined', document_name='strojirenska_firma')

🚀 Evaluating: strojirenska_firma
   Otázky: 12 | Metody: 3 | Celkem LLM volání: 36
   Timeout: 120s | Mode: combined

[1/12] Jake jsou klíčové aktivity v evidenci TPV?
  ⏳ Naive RAG... ✅ 23.6s
    → Klíčové aktivity v evidenci TPV (Technické přípravy výroby) jsou:  1.  **Prověření a analýza požadavků na výrobu**: Prov...
    BERT F1: 0.675 | ROUGE-L R: 0.211 | ROUGE-L F1: 0.121
  ⏳ Graph RAG... 🔍 Nalezené entity pro graf: ['klíčové aktivity', 'evidencia TPV']
🎯 Přímý fulltext dotaz do grafu: *jake* OR *jsou* OR *klíčo* OR *aktiv* OR *evide* OR *tpv*
  🎯 ZÁCHYT V GRAFU: Nalezen uzel [Kapitola] s názvem '15. Plánování a koordinace výrobních zakázek | 15.1 Přehled a obsah úloh plánování a koordinace výrobních zakázek | 15.1.4 Analýzy výrobních zakázek | 15.1.4.1 Klíčové aktivity'
  🎯 ZÁCHYT V GRAFU: Nalezen uzel [Kapitola] s názvem '16. Technická příprava, výroby, TPV | 16.1 Přehled a obsah úloh TPV | 16.1.1 Evidence technické přípravy výroby, TPV'
  🎯 ZÁCHYT V GRAFU: Nalezen uzel [Kapito

/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_55056/1246177949.py:192: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm_judge)
/var/folders/bq/dvsycyn560sgx_gql985ch9w0000gn/T/ipykernel_55056/1246177949.py:193: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(embeddings)


Evaluating:   0%|          | 0/144 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[4]: OutputParserException(Invalid json output: Klíčové aktivity v evidenci TPV jsou:
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )



📊 VÝSLEDKY PER OTÁZKA — strojirenska_firma


,method,question,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_L_P,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
0,Naive RAG,Jake jsou klíčové aktivity v evidenci TPV?,0.660,0.691,0.675,0.085,0.211,0.121,0.800,0.532,0.500,0.172
1,Graph RAG,Jake jsou klíčové aktivity v evidenci TPV?,0.651,0.769,0.705,0.138,1.000,0.243,NaN,0.883,1.000,0.417
2,Hybrid GraphRAG,Jake jsou klíčové aktivity v evidenci TPV?,0.671,0.812,0.735,0.220,0.915,0.354,0.917,0.517,1.000,0.685
3,Naive RAG,Jaké funkce zahrnuje pokročilá analytika v mar...,0.667,0.800,0.727,0.188,0.569,0.283,1.000,0.644,1.000,0.957
4,Graph RAG,Jaké funkce zahrnuje pokročilá analytika v mar...,0.617,0.665,0.640,0.067,0.176,0.097,0.833,0.467,0.000,0.173
5,Hybrid GraphRAG,Jaké funkce zahrnuje pokročilá analytika v mar...,0.662,0.766,0.711,0.176,0.451,0.253,1.000,0.619,1.000,0.746
6,Naive RAG,Jaké jsou klíčové aktivity analytické úlohy v ...,0.613,0.649,0.630,0.073,0.164,0.101,1.000,0.678,0.857,0.925
7,Graph RAG,Jaké jsou klíčové aktivity analytické úlohy v ...,0.761,0.872,0.813,0.510,0.891,0.649,1.000,0.691,1.000,0.837
8,Hybrid GraphRAG,Jaké jsou klíčové aktivity analytické úlohy v ...,0.669,0.806,0.731,0.200,0.945,0.330,0.091,0.678,1.000,0.599
9,Naive RAG,Jaký je účel TPV v technické přípravě?,0.661,0.738,0.698,0.118,0.400,0.183,0.909,0.501,1.000,0.963



📊 PRŮMĚRNÉ SKÓRE — strojirenska_firma


,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_L_P,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
method,,,,,,,,,,
Graph RAG,0.695,0.812,0.748,0.285,0.792,0.372,0.861,0.629,0.833,0.689
Hybrid GraphRAG,0.745,0.862,0.798,0.416,0.862,0.527,0.917,0.596,1.000,0.756
Naive RAG,0.683,0.724,0.701,0.249,0.342,0.243,0.892,0.558,0.785,0.594



🏆 Nejlepší ROUGE-L Recall (úplnost):       Hybrid GraphRAG
🏆 Nejlepší BERTScore F1 (sémantika):        Hybrid GraphRAG
🏆 Nejlepší Answer Correctness (RAGAS):      Hybrid GraphRAG

✅ Výsledky uloženy → evaluation_strojirenska_firma.csv


,document,method,question,answer,context,time_s,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_L_P,ROUGE_L_R,ROUGE_L_F1,faithfulness,answer_relevancy,context_recall,answer_correctness
0,strojirenska_firma,Naive RAG,Jake jsou klíčové aktivity v evidenci TPV?,Klíčové aktivity v evidenci TPV (Technické pří...,Účelem je vytvářet a zajistit průběžnou operat...,23.6,0.660134,0.691110,0.675267,0.084746,0.211268,0.120968,0.800000,0.531999,0.500000,0.171632
1,strojirenska_firma,Graph RAG,Jake jsou klíčové aktivity v evidenci TPV?,Klíčové aktivity v evidenci TPV (Technické pří...,### DOKUMENTY Z GRAFU:\nÚčelem úlohy je vytvář...,34.7,0.650801,0.768885,0.704932,0.138132,1.000000,0.242735,NaN,0.883478,1.000000,0.416898
2,strojirenska_firma,Hybrid GraphRAG,Jake jsou klíčové aktivity v evidenci TPV?,Klíčové aktivity v evidenci TPV (Technické pří...,### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚ...,28.8,0.671422,0.812179,0.735123,0.219595,0.915493,0.354223,0.916667,0.516861,1.000000,0.684832
3,strojirenska_firma,Naive RAG,Jaké funkce zahrnuje pokročilá analytika v mar...,Pokročilá analytika v marketingu zahrnuje násl...,- Jaké klíčové funkce a aktivity má zahrnovat ...,22.7,0.667100,0.799589,0.727360,0.188312,0.568627,0.282927,1.000000,0.643843,1.000000,0.957472
4,strojirenska_firma,Graph RAG,Jaké funkce zahrnuje pokročilá analytika v mar...,"Není možné odpovědět na otázku, protože se týk...",### DOKUMENTY Z GRAFU:\nÚčelem je řešit finanč...,15.1,0.616820,0.665422,0.640200,0.066667,0.176471,0.096774,0.833333,0.467438,0.000000,0.173291
5,strojirenska_firma,Hybrid GraphRAG,Jaké funkce zahrnuje pokročilá analytika v mar...,Pokročilá analytika v marketingu zahrnuje násl...,### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚ...,17.9,0.662463,0.766282,0.710601,0.175573,0.450980,0.252747,1.000000,0.619493,1.000000,0.746291
6,strojirenska_firma,Naive RAG,Jaké jsou klíčové aktivity analytické úlohy v ...,Klíčovými aktivitami analytické úlohy v contro...,Jako podstatné výstupy z controllingu jsou: \...,17.7,0.612903,0.648730,0.630308,0.072581,0.163636,0.100559,1.000000,0.677789,0.857143,0.924571
7,strojirenska_firma,Graph RAG,Jaké jsou klíčové aktivity analytické úlohy v ...,Klíčové aktivity analytické úlohy v controllin...,### DOKUMENTY Z GRAFU:\nÚčelem analytické úloh...,16.8,0.761158,0.871941,0.812792,0.510417,0.890909,0.649007,1.000000,0.691226,1.000000,0.836733
8,strojirenska_firma,Hybrid GraphRAG,Jaké jsou klíčové aktivity analytické úlohy v ...,Analytická úloha v kontrollingu zahrnuje kompl...,### 1. DOKUMENTY Z GRAFU (PRIMÁRNÍ ZDROJ - ČTĚ...,24.7,0.668770,0.805842,0.730935,0.200000,0.945455,0.330159,0.090909,0.678099,1.000000,0.598735
9,strojirenska_firma,Naive RAG,Jaký je účel TPV v technické přípravě?,Účelem Technických postupů výroby (TPV) v tech...,Účelem analytické úlohy je především dosažení ...,13.9,0.661023,0.738357,0.697553,0.118227,0.400000,0.182510,0.909091,0.501090,1.000000,0.962875
